# Week 4 Day 3 — Causal Self-Attention / Decoder Block
**Jul 22, 2026**

Week 3 Day 6's Try Yourself flagged causal masking; today it's the main event. A model that predicts the next token from previous ones can't be allowed to "see" the answer sitting right there in the input sequence — causal masking is what prevents that. Structurally, a GPT-style decoder block *is* an `EncoderBlock` with one addition: a mask blocking every position from attending to anything after itself.

Scaffold as usual: TODO stubs, hints not formulas, self-check cells.

## Part 1: Setup

Given. Toy sequence data, same shape as previous days.

In [1]:
import torch
import torch.nn as nn
import math

torch.manual_seed(0)
batch, seq_len, d_model, num_heads, d_ff = 1, 5, 8, 2, 16

x = torch.randn(batch, seq_len, d_model)
print(x.shape)

torch.Size([1, 5, 8])


## Part 2: Build the causal mask

TODO: implement `make_causal_mask(seq_len)`, returning a `(seq_len, seq_len)` boolean tensor where `True` means "blocked" (query position can't attend to this key position) — specifically, position `i` should be blocked from attending to any position `j > i`.

`torch.triu` (upper triangle) is the relevant building block — think about which `diagonal` argument value excludes the diagonal itself (position `i` attending to itself, `j == i`, should always be *allowed*, not blocked).

In [2]:
def make_causal_mask(seq_len):
    # TODO
    return torch.triu(torch.ones(seq_len, seq_len, dtype=torch.bool),
                      diagonal=1
    
    )
    
    raise NotImplementedError

mask = make_causal_mask(seq_len)
assert mask.shape == (seq_len, seq_len)
assert mask.dtype == torch.bool
assert not mask.diagonal().any(), "a position shouldn't be masked from attending to itself"
assert mask[0, 1:].all(), "position 0 should be blocked from every later position"
assert not mask[-1].any(), "the last position should see everything (nothing comes after it)"
print(mask)
print("causal mask OK")

tensor([[False,  True,  True,  True,  True],
        [False, False,  True,  True,  True],
        [False, False, False,  True,  True],
        [False, False, False, False,  True],
        [False, False, False, False, False]])
causal mask OK


## Part 3: Masked multi-head attention

TODO: build `MultiHeadAttention(d_model, num_heads)` — identical to Week 3 Day 7's version, but `forward(x, mask=None)` now accepts an optional mask and applies it right before the `softmax`.

Where the mask plugs in: after computing raw scores (`Q @ K^T / sqrt(head_dim)`) but before `softmax`, use `scores.masked_fill(mask, float('-inf'))` wherever `mask` is `True`. Setting a score to `-inf` guarantees `softmax` sends its weight to exactly `0` — an already-computed near-zero probability isn't strong enough; not being computed *at all* is what a hard boundary requires.

In [9]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads
        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)

    def forward(self, x, mask=None):
        # TODO: same as week3 day7's MultiHeadAttention, but apply `mask` to scores before softmax
        batch, seq_len, d_model = x.shape
        
        #(batch, seq_len, d_model) -> (batch, seq_len, num_heads, head_dim)
        Q = self.q_proj(x)
        K = self.k_proj(x)
        V = self.v_proj(x)
        
        # (batch, num_heads, seq_len, head_dim)
        Q = Q.view(batch, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        K = K.view(batch, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        V = V.view(batch, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        
        #batch, num_heads, seq_len, seq_len
        scores = Q @ K.transpose(-2, -1) / math.sqrt(self.head_dim) #
        if mask is not None:
            # mask shape: (seq_len, seq_len) -> (1, 1, seq_len, seq_len)
            mask = mask.unsqueeze(0).unsqueeze(0)
            scores = scores.masked_fill(mask, float('-inf'))
        
        weights = torch.softmax(scores, dim=-1)
        attention = weights @ V  # (batch, num_heads, seq_len, head_dim
        
        #combine all heads:
        #(batch, seq_len, num_heads, head_dim) -> (batch, seq_len, d_model)
        attention = attention.transpose(1, 2).contiguous()
        
        #batch, seq_len, d_model
        attention = attention.view(batch, seq_len, d_model)
        out = self.out_proj(attention)
        return out, weights

mha = MultiHeadAttention(d_model, num_heads)
out, weights = mha(x, mask=mask)

assert out.shape == (batch, seq_len, d_model)
assert weights.shape == (batch, num_heads, seq_len, seq_len)

upper_triangle = torch.triu(weights, diagonal=1)
assert upper_triangle.max().item() == 0.0, "masked positions should have exactly zero attention weight"
row_sums = weights.sum(dim=-1)
assert torch.allclose(row_sums, torch.ones_like(row_sums), atol=1e-5), "each row should still sum to 1 over its allowed positions"
print("masked attention OK -- zero weight on future positions, still sums to 1")

masked attention OK -- zero weight on future positions, still sums to 1


## Part 4: Check against `nn.MultiheadAttention`'s `attn_mask`

Given. PyTorch's built-in accepts an `attn_mask` argument with the exact same "`True`/`-inf` means blocked" convention used above.

In [10]:
mha.eval()
with torch.no_grad():
    my_out, _ = mha(x, mask=mask)

ref = nn.MultiheadAttention(d_model, num_heads, batch_first=True)
ref.eval()
with torch.no_grad():
    ref.in_proj_weight.copy_(torch.cat([mha.q_proj.weight, mha.k_proj.weight, mha.v_proj.weight], dim=0))
    ref.in_proj_bias.copy_(torch.cat([mha.q_proj.bias, mha.k_proj.bias, mha.v_proj.bias], dim=0))
    ref.out_proj.weight.copy_(mha.out_proj.weight)
    ref.out_proj.bias.copy_(mha.out_proj.bias)
    ref_out, _ = ref(x, x, x, attn_mask=mask, need_weights=False)

assert torch.allclose(my_out, ref_out, atol=1e-4), "doesn't match nn.MultiheadAttention with attn_mask"
print("matches nn.MultiheadAttention's attn_mask behavior")

matches nn.MultiheadAttention's attn_mask behavior


## Part 5: `DecoderBlock`

TODO: build `DecoderBlock(d_model, num_heads, d_ff)` — structurally identical to Week 3 Day 7's `EncoderBlock` (residual + `LayerNorm` around attention, residual + `LayerNorm` around a feedforward network), except its `forward` builds a causal mask from the input's `seq_len` and passes it into the attention sublayer.

This is the entire architectural difference between an encoder block and a GPT-style decoder block used without cross-attention: whether a mask is applied. Everything else — the feedforward network, the residual connections, the layer norms — is identical.

In [11]:
class DecoderBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff):
        super().__init__()
        # TODO: self.attn, self.norm1, self.ff, self.norm2 (same as EncoderBlock)
        self.attn = MultiHeadAttention(d_model, num_heads)
        self.norm1 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Linear(d_ff, d_model)
        )
        self.norm2 = nn.LayerNorm(d_model)
        

    def forward(self, x):
        # TODO: build a causal mask for x's seq_len, pass it into self.attn,
        seq_len = x.size(1)
        
        # Move the mask to the same device as x
        mask = make_causal_mask(seq_len).to(x.device)
        
        #masked attention
        attn_out, _ = self.attn(x, mask=mask)
        
        # First residual connection and normalization
        x = self.norm1(x + attn_out)
        
        #Feed forward network
        ff_out = self.ff(x)
        
        #second residual connection and normalization
        out = self.norm2(x + ff_out)
        
        return out


block = DecoderBlock(d_model, num_heads, d_ff)
out = block(x)
assert out.shape == (batch, seq_len, d_model), f"expected {(batch, seq_len, d_model)}, got {tuple(out.shape)}"
print("DecoderBlock shape OK")

DecoderBlock shape OK


## Try yourself

1. Change one value in `x` at the *last* position and rerun `block(x)` -- confirm every output position changes (since every position can see the last one bidirectionally in a real encoder, but here only the last position's own output should be free to depend on it directly through attention -- trace through which output positions the change should and shouldn't reach).
2. Stack 3 `DecoderBlock`s and confirm shapes still work at depth, same as Week 4 Day 2's encoder stack.
3. Add `is_causal=True` to `F.scaled_dot_product_attention` instead of building an explicit mask tensor, and confirm it produces the same output as your `make_causal_mask` approach -- PyTorch has a dedicated fast path for exactly this common case.
4. What happens if you accidentally pass `float(0)` into `masked_fill` instead of `float('-inf')`? Try it and look at the resulting attention weights -- why does a hard `-inf` matter here rather than "just make it small"?